## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240318'
end_date = '20240331'

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/city/bangalore/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/city/bangalore/


##usecase_tag

usecase_tag_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id,
        order_id,
        drop_location_hex_10 hex_10
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host = 'android'
        AND city_name = 'Bangalore'
    GROUP BY 1,2,3
    ), 

    use_case AS (
    
    SELECT
        hex_10, 
        usecase_tag
    FROM
        (
        SELECT 
            hex_10, 
            combined_final_usecase_accuracy as usecase_tag,
            ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
        FROM
            hive.experiments_internal.combined_usecase_hex_10_level
        WHERE 
            aoi = 'Bangalore, India'
        )
    WHERE   
        seq_no = 1
    ),
    
    merge AS (
    SELECT
        customer_id,
        usecase_tag,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY orders DESC) seq_no
    FROM 
        (
        SELECT
            a.customer_id,
            COALESCE(b.usecase_tag, 'Unknown') usecase_tag,
            COUNT(DISTINCT order_id) orders
        FROM 
            active_customers a
        LEFT JOIN 
            use_case b
            ON a.hex_10 = b.hex_10
        GROUP BY 1,2
        )
    WHERE 
        usecase_tag != 'Unknown'
    )
    
    SELECT
        a.customer_id,
        COALESCE(b.usecase_tag, 'Unknown') usecase_tag
    FROM 
        active_customers a
    LEFT JOIN 
        merge b 
        ON a.customer_id = b.customer_id
        AND seq_no = 1
    GROUP BY 1,2

"""

df_usecase_tag_query = pd.read_sql(usecase_tag_query, connection)
df_usecase_tag_query.to_csv(local_datasets + 'raw/usecase_tag_v1.csv', index=False)

In [7]:
df_usecase_tag = pd.read_csv(local_datasets + 'raw/usecase_tag_v1.csv')
print(df_usecase_tag.shape)
df_usecase_tag.head(2)

(1150046, 2)


,customer_id,usecase_tag
0,6215c014bdbee2aae67bb674,household_needs
1,6446114b13a79336a303e974,transit_station


### Active customer (RR-customers)

In [83]:
df_bangalore_active_customer = pd.read_csv(local_datasets + 'raw/bangalore_customers_v1.csv')
df_bangalore_active_customer = df_bangalore_active_customer.drop('app_list_set', axis=1)
print(df_bangalore_active_customer.shape)
df_bangalore_active_customer.head(2)

(1024954, 16)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,63830d63b0fd068a5918609f,MALE,497.0,NaN,2.0,1.0,1.0,PHH,6,HOOK,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,1.0,b. low rpc
1,62a5ec437d7967150de68477,MALE,665.0,NaN,2.0,1.0,1.0,PHH,98,HOOK,HIGH_INCOME,ONLY_AUTO,UNKNOWN,NPS,1.0,b. low rpc


In [84]:
df_bangalore_active_customer = pd.merge(df_bangalore_active_customer, df_usecase_tag,
                                        how='left', on=['customer_id'])

### customer installed apps

In [85]:
df_customer_mapped = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapped_v1.csv')
print(df_customer_mapped.shape)
df_customer_mapped.head(2)

(1024224, 14)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,5737c6aeddbec2203f733176,"['ms edge', 'onedrive', 'instagram', 'microsof...",17,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'D...",9,"['Finance_Investment', 'Ride haling', 'Office'...",4,0,0,0,1,1,1,1
1,5737c6e3ddbec2203f733341,"['axis mobile', 'instagram', 'facebook', 'flip...",12,"['Tools', 'Social', 'OTT', 'Streaming_Music', ...",7,"['Ride haling', 'Rest']",2,0,0,0,0,0,1,1


### Customer app & cat explode mapping

In [86]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapping_v1.csv')
df_customer_cat_mapping = df_customer_app_cat_mapping[['customer_id', 
#                                                        'categories_l1', 
                                                       'categories_l2']]\
                            .drop_duplicates()
df_customer_cat_mapping.head(1)

,customer_id,categories_l2
0,63830d63b0fd068a5918609f,Rest


In [87]:
total_customers = df_customer_cat_mapping.customer_id.nunique()
total_customers

1024224

In [88]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['Delivery_Food' 'OTT' 'Ecommerce' 'Office' 'Tools' 'Travel_Ridehailing'
 'Social' 'Entertainment' 'News' 'Streaming_Music' 'Delivery_Grocery'
 'Finance_Transactions' 'Dating' 'Finance_Investment' 'Educational'
 'Travel_Bookings' 'Wearable' 'Gaming' 'Health' 'Driver_App'
 'Finance_News' 'Fantasy_Sports' 'Devotional']


In [89]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Rest' 'Office' 'Ride haling' 'Finance_Investment' 'Educational' 'Gaming'
 'Driver_App']


In [90]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [91]:
### Office
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,ola,604435,59.01
1,Ride haling,uber,521605,50.93
2,Ride haling,namma yatri,231295,22.58
3,Ride haling,quick ride,30502,2.98
4,Ride haling,blusmart,26631,2.60
5,Ride haling,in drive,25496,2.49
6,Ride haling,driveu driver app,3213,0.31
7,Ride haling,quickride,1185,0.12
8,Ride haling,jugnoo,1060,0.10
9,Office,zoom,277846,27.13


### Merge raw data

In [92]:
df_bangalore_active_customer

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag
0,63830d63b0fd068a5918609f,MALE,497.0,NaN,2.0,1.0,1.0,PHH,6,HOOK,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,1.0,b. low rpc,educational
1,62a5ec437d7967150de68477,MALE,665.0,NaN,2.0,1.0,1.0,PHH,98,HOOK,HIGH_INCOME,ONLY_AUTO,UNKNOWN,NPS,1.0,b. low rpc,residential
2,60f1801d8f347996e117f370,MALE,996.0,24.0,53.0,34.0,17.0,PHH,177,COMMITTED,UNKNOWN,LINK_AUTO,UNKNOWN,NPS,17.0,d. high rpc,residential
3,64ded9f6860b137a546000ff,MALE,155.0,NaN,28.0,14.0,10.0,PHH,92,COMMITTED,MEDIUM_INCOME,ONLY_CAB,LONG_DISTANCE,PS,10.0,d. high rpc,residential
4,61dc6c94d80f2b17baa43140,MALE,818.0,NaN,7.0,3.0,2.0,PHH,180,HOOK,UNKNOWN,NO_AFFINITY,UNKNOWN,PS,2.0,c. medium rpc,residential
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1024949,63230bcd5d3af40420020164,FEMALE,570.0,NaN,2.0,1.0,1.0,PHH,76,HOOK,HIGH_INCOME,ONLY_AUTO,MEDIUM_DISTANCE,NPS,1.0,b. low rpc,residential
1024950,6337ad2d5366695b29261b1c,MALE,554.0,33.0,1.0,1.0,1.0,PHH,8,HOOK,HIGH_INCOME,ONLY_AUTO,UNKNOWN,PS,1.0,b. low rpc,Unknown
1024951,61a740ec41b9b042201170a6,MALE,858.0,NaN,2.0,1.0,0.0,PHH,32,INACTIVE,MEDIUM_INCOME,ONLY_LINK,UNKNOWN,UNKNOWN,NaN,a. zero rpc,transit_station
1024952,6583421465ece0023043790b,MALE,108.0,NaN,2.0,1.0,1.0,PHH,8,HOOK,HIGH_INCOME,ONLY_AUTO,UNKNOWN,NPS,1.0,b. low rpc,educational


In [93]:
df_bangalore_active_customer[df_bangalore_active_customer['customer_id'].isin(['63230bcd5d3af40420020164'])]

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag
1024949,63230bcd5d3af40420020164,FEMALE,570.0,NaN,2.0,1.0,1.0,PHH,76,HOOK,HIGH_INCOME,ONLY_AUTO,MEDIUM_DISTANCE,NPS,1.0,b. low rpc,residential


In [94]:
df_customer_mapped

,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,5737c6aeddbec2203f733176,"['ms edge', 'onedrive', 'instagram', 'microsof...",17,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'D...",9,"['Finance_Investment', 'Ride haling', 'Office'...",4,0,0,0,1,1,1,1
1,5737c6e3ddbec2203f733341,"['axis mobile', 'instagram', 'facebook', 'flip...",12,"['Tools', 'Social', 'OTT', 'Streaming_Music', ...",7,"['Ride haling', 'Rest']",2,0,0,0,0,0,1,1
2,5737c6fdddbec2203f733422,"['telegram', 'instagram', 'linkedin', 'groww',...",23,"['Tools', 'Social', 'News', 'Delivery_Food', '...",11,"['Finance_Investment', 'Ride haling', 'Office'...",4,0,0,0,1,1,1,1
3,5737c6ffddbec2203f733437,"['telegram', 'instagram', 'google sheets', 'fa...",17,"['Tools', 'Social', 'News', 'Delivery_Food', '...",9,"['Ride haling', 'Rest']",2,0,0,0,0,0,1,1
4,5737c70bddbec2203f7334a0,"['google news', 'wynk music', 'facebook', 'pay...",7,"['Tools', 'Social', 'News', 'Gaming', 'Streami...",6,"['Gaming', 'Rest']",2,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1024219,6609a98f75d287202ff3d68b,"['zee5', 'myntra', 'telegram', 'instagram', 'f...",12,"['Social', 'News', 'OTT', 'Ecommerce', 'Stream...",6,"['Driver_App', 'Rest']",2,1,0,0,0,0,0,1
1024220,6609a9f3c1792aa62702a13b,"['facebook', 'google news', 'flipkart', 'messe...",8,"['Social', 'News', 'Streaming_Music', 'Finance...",6,"['Ride haling', 'Rest']",2,0,0,0,0,0,1,1
1024221,6609aa278fcbab0f6302fbe0,"['onedrive', 'microsoft 365', 'samsung galaxy ...",13,"['Tools', 'Social', 'OTT', 'Streaming_Music', ...",9,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1
1024222,6609aab4db0e9caafc405cd8,"['onedrive', 'amazon prime video', 'instagram'...",15,"['Tools', 'Social', 'OTT', 'Streaming_Music', ...",8,"['Office', 'Rest']",2,0,0,0,1,0,0,1


In [95]:
df_customer_data = pd.merge(df_bangalore_active_customer, df_customer_mapped,
                            how='inner', on=['customer_id']
                           )
print(df_customer_data.shape)
df_customer_data.head(1)

(1024224, 30)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,63830d63b0fd068a5918609f,MALE,497.0,NaN,2.0,1.0,1.0,PHH,6,HOOK,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,1.0,b. low rpc,educational,"['telegram', 'instagram', 'zoom', 'zomato', 'l...",23,"['Tools', 'Social', 'News', 'Delivery_Food', '...",10,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1


#### Derived columns

In [96]:
## RPC

df_customer_data['rpc'] =  df_customer_data['net_count'].replace(0, np.nan)
df_customer_data.rpc.describe()

count    848197.000000
mean          2.934055
std           3.443234
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max         104.000000
Name: rpc, dtype: float64

In [97]:
df_customer_data['rpc_bucket'] = np.where(df_customer_data['net_count'] < 1 , 'a. ZERO',
                                    np.where(df_customer_data['net_count'] < 2 , 'b. LOW',
                                        np.where(df_customer_data['net_count'] < 4 , 'c. MEDIUM', 
                                            np.where(df_customer_data['net_count'] >= 4 , 'd. HIGH', None))))

In [98]:
## app_count_bucket

df_customer_data.app_count.describe()

count    1.024224e+06
mean     1.973654e+01
std      9.985884e+00
min      1.000000e+00
25%      1.200000e+01
50%      1.800000e+01
75%      2.600000e+01
max      8.400000e+01
Name: app_count, dtype: float64

In [99]:
df_customer_data['app_count_bucket'] = np.where(df_customer_data['net_count'] < 5 , '1-5',
                                        np.where(df_customer_data['net_count'] < 10 , '6-10',
                                          np.where(df_customer_data['net_count'] < 16 , '11-15', 'Above 15')))

In [100]:
## category_count_bucket

df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.category_count.describe()

count    1.024224e+06
mean     3.163860e+00
std      1.258072e+00
min      1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      7.000000e+00
Name: category_count, dtype: float64

In [101]:
df_customer_data['category_count_bucket'] = np.where(df_customer_data['category_count'] < 2 , '1',
                                              np.where(df_customer_data['category_count'] < 3 , '2',
                                                np.where(df_customer_data['category_count'] < 4 , '3','Above 3')))

In [102]:
## category_count

# df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.rapido_age.describe()

count    1.022299e+06
mean     6.896744e+02
std      6.017760e+02
min      1.000000e+00
25%      2.170000e+02
50%      5.610000e+02
75%      8.950000e+02
max      2.876000e+03
Name: rapido_age, dtype: float64

#### Merge

In [103]:
df_customer_data_explode = pd.merge(df_customer_data,
                                    df_customer_cat_mapping,
                                    how='inner', on =['customer_id'])

df_customer_data_explode[df_customer_data_explode['customer_id'] == '63230bcd5d3af40420020164'].head()

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2_x,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest,rpc_bucket,app_count_bucket,category_count,category_count_bucket,categories_l2_y
3240482,63230bcd5d3af40420020164,FEMALE,570.0,NaN,2.0,1.0,1.0,PHH,76,HOOK,HIGH_INCOME,ONLY_AUTO,MEDIUM_DISTANCE,NPS,1.0,1.0,residential,"['instagram', 'amazon music', 'zomato', 'linke...",20,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'S...",10,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1,b. LOW,1-5,3,3,Ride haling
3240483,63230bcd5d3af40420020164,FEMALE,570.0,NaN,2.0,1.0,1.0,PHH,76,HOOK,HIGH_INCOME,ONLY_AUTO,MEDIUM_DISTANCE,NPS,1.0,1.0,residential,"['instagram', 'amazon music', 'zomato', 'linke...",20,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'S...",10,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1,b. LOW,1-5,3,3,Rest
3240484,63230bcd5d3af40420020164,FEMALE,570.0,NaN,2.0,1.0,1.0,PHH,76,HOOK,HIGH_INCOME,ONLY_AUTO,MEDIUM_DISTANCE,NPS,1.0,1.0,residential,"['instagram', 'amazon music', 'zomato', 'linke...",20,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'S...",10,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1,b. LOW,1-5,3,3,Office


## User Base Distribution analysis

LTR-Segment

Service Affinity

Distance Affinity

Customers Use Case

AO- Net Funnel

Gender

Age (Whatever fill rate we have)

Income

RPC

In [104]:
tot_customers = df_customer_cat_mapping.customer_id.nunique()
df_category_agg = df_customer_cat_mapping\
                    .groupby(['categories_l2'])\
                    .agg(total_customers = ('customer_id','nunique'))\
                    .sort_values(['total_customers'], ascending=False)\
                    .reset_index()
df_category_agg['% distribution'] =  (df_category_agg['total_customers']*100.00/tot_customers).round(2)
df_category_agg

,categories_l2,total_customers,% distribution
0,Rest,1023779,99.96
1,Ride haling,763669,74.56
2,Office,606538,59.22
3,Finance_Investment,377134,36.82
4,Educational,215925,21.08
5,Gaming,152769,14.92
6,Driver_App,100687,9.83


#### LTR-Segment

In [105]:
ltr_segment_city = df_customer_data\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ltr_segment_city['% city threshold'] =  (ltr_segment_city['customers']*100.00/ltr_segment_city.customers.sum()).round(2)
ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,187941,18.35
1,LTR 0,27762,2.71
2,PHH,808213,78.91
3,UNKNOWN,308,0.03


In [106]:
df1 = df_customer_data_explode[~df_customer_data_explode['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers'])

customers                                                 \
categories_l2 Driver_App Educational Finance_Investment  Gaming  Office   
ltr_segment                                                               
HH                 20636       31852              48738   27945   87388   
LTR 0               3586        4425               6336    4315   11373   
PHH                76429      179589             321941  120479  507582   

                                   
categories_l2    Rest Ride haling  
ltr_segment                        
HH             187823      112889  
LTR 0           27738       15799  
PHH            807910      634746

In [107]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customer %'])

In [108]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
ltr_segment                                                                     
HH                  10.98       16.95              25.93  14.87  46.50  99.94   
LTR 0               12.92       15.94              22.82  15.54  40.97  99.91   
PHH                  9.46       22.22              39.83  14.91  62.80  99.96   

                           
categories_l2 Ride haling  
ltr_segment                
HH                  60.07  
LTR 0               56.91  
PHH                 78.54

#### Other testing

In [109]:
df_test = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l2'].isin(['Office', 
                                                                                         'Educational',
                                                                                         'Ride haling'
                                                                                        ])]
df_test = pd.merge(df_test,df_bangalore_active_customer[['customer_id', 'ltr_segment']], how='inner', on='customer_id')
df_office = df_test[df_test['categories_l2'].isin(['Office'])]
df_education = df_test[df_test['categories_l2'].isin(['Educational'])]
df_ride_hailing = df_test[df_test['categories_l2'].isin(['Ride haling'])]

##### Explode - Ride Hailing

In [110]:
rh_ltr_segment_city = df_ride_hailing\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rh_ltr_segment_city['% city threshold'] =  (rh_ltr_segment_city['customers']*100.00/rh_ltr_segment_city.customers.sum()).round(2)
rh_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,112889,14.78
1,LTR 0,15799,2.07
2,PHH,634746,83.12
3,UNKNOWN,235,0.03


In [111]:
df1 = df_ride_hailing[~df_ride_hailing['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
ride_hailing_total_customers = df_ride_hailing.customer_id.nunique()
df2 = pd.merge(df1,rh_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/ride_hailing_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                        \
app_name     blusmart driveu driver app in drive jugnoo namma yatri     ola   
ltr_segment                                                                   
HH               1776               380     2977     94       22721   86401   
LTR 0             261                48      605     17        3294   11790   
PHH             24587              2785    21907    949      205220  506056   

                                          
app_name    quick ride quickride    uber  
ltr_segment                               
HH                2322       326   67003  
LTR 0              313        91    9398  
PHH              27861       767  445047

In [112]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [113]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                       \
app_name       blusmart driveu driver app in drive jugnoo namma yatri    ola   
ltr_segment                                                                    
HH                 1.57              0.34     2.64   0.08       20.13  76.54   
LTR 0              1.65              0.30     3.83   0.11       20.85  74.62   
PHH                3.87              0.44     3.45   0.15       32.33  79.73   

                                         
app_name    quick ride quickride   uber  
ltr_segment                              
HH                2.06      0.29  59.35  
LTR 0             1.98      0.58  59.48  
PHH               4.39      0.12  70.11

In [114]:
# namma yatri
# ola 
# uber 

##### Explode - Office

In [115]:
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
0,63830d63b0fd068a5918609f,zoom,zoom,Office,Office,PHH
3,62a5ec437d7967150de68477,webex,webex,Office,Office,PHH


In [116]:
def assign_office_value(row):
    if row['app_name'] in ['asana','cisco','confluence','github','google analytics',
                           'intune  company portal','jira','miro','slack','trello','zoho mail','zoho meeting']:
        return 'code_office_app'
    else:
        return 'secondary_office_app'

df_office['app_name_tag'] = df_office.apply(assign_office_value, axis=1)
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
0,63830d63b0fd068a5918609f,zoom,zoom,Office,Office,PHH,secondary_office_app
3,62a5ec437d7967150de68477,webex,webex,Office,Office,PHH,secondary_office_app


In [117]:
df_office[df_office['app_name_tag'] == 'code_office_app'].app_name.unique()

array(['intune  company portal', 'zoho meeting', 'slack', 'github',
       'trello', 'miro', 'zoho mail', 'cisco', 'jira', 'google analytics',
       'confluence', 'asana'], dtype=object)

In [118]:
office_ltr_segment_city = df_office\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
office_ltr_segment_city['% city threshold'] =  (office_ltr_segment_city['customers']*100.00/office_ltr_segment_city.customers.sum()).round(2)
office_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,87388,14.41
1,LTR 0,11373,1.88
2,PHH,507582,83.69
3,UNKNOWN,195,0.03


In [119]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                      \
app_name        asana   cisco confluence dropbox  github google analytics   
ltr_segment                                                                 
HH               93.0   155.0       78.0  1242.0  1003.0            101.0   
LTR 0            19.0    15.0       17.0   166.0   110.0             17.0   
PHH            1263.0  1098.0     1351.0  9596.0  8020.0           1425.0   

                                                                               \
app_name    google authenticator intune  company portal    jira microsoft 365   
ltr_segment                                                                     
HH                        7792.0                14976.0   402.0       31623.0   
LTR 0                      886.0                 1703.0    54.0        4065.0   
PHH                      66351.0               114168.0  4975.0      192046.0   

                                                                       \
app_name    microsoft teams   miro ms authenticator nishtha  onedrive   
ltr_segment                                                             
HH                  31960.0   50.0          18708.0     NaN   37642.0   
LTR 0                3757.0    7.0           2193.0     1.0    5130.0   
PHH                222839.0  554.0         146093.0     NaN  200604.0   

                                                                   \
app_name      outlook  pocket    slack  trello    webex zoho mail   
ltr_segment                                                         
HH            34834.0   146.0   3939.0   194.0   5295.0     637.0   
LTR 0          4448.0    18.0    487.0    20.0    625.0      65.0   
PHH          225291.0  1986.0  43282.0  2142.0  34751.0    5624.0   

                                    
app_name    zoho meeting      zoom  
ltr_segment                         
HH                 307.0   37964.0  
LTR 0               41.0    5013.0  
PHH               2133.0  234757.0

In [120]:
#### print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

customers %                                                   \
app_name          asana cisco confluence dropbox github google analytics   
ltr_segment                                                                
HH                 0.11  0.18       0.09    1.42   1.15             0.12   
LTR 0              0.17  0.13       0.15    1.46   0.97             0.15   
PHH                0.25  0.22       0.27    1.89   1.58             0.28   

                                                                             \
app_name    google authenticator intune  company portal  jira microsoft 365   
ltr_segment                                                                   
HH                          8.92                  17.14  0.46         36.19   
LTR 0                       7.79                  14.97  0.47         35.74   
PHH                        13.07                  22.49  0.98         37.84   

                                                                             \
app_name    microsoft teams  miro ms authenticator nishtha onedrive outlook   
ltr_segment                                                                   
HH                    36.57  0.06            21.41     NaN    43.07   39.86   
LTR 0                 33.03  0.06            19.28    0.01    45.11   39.11   
PHH                   43.90  0.11            28.78     NaN    39.52   44.39   

                                                                     
app_name    pocket slack trello webex zoho mail zoho meeting   zoom  
ltr_segment                                                          
HH            0.17  4.51   0.22  6.06      0.73         0.35  43.44  
LTR 0         0.16  4.28   0.18  5.50      0.57         0.36  44.08  
PHH           0.39  8.53   0.42  6.85      1.11         0.42  46.25

In [121]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                     20047                86149
LTR 0                   2295                11235
PHH                   164517               497763

In [122]:
#### print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

customers %                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                     22.94                98.58
LTR 0                  20.18                98.79
PHH                    32.41                98.07

##### Explode - Education

In [123]:
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
8,60f1801d8f347996e117f370,moodle,moodle,Educational,Educational,PHH
16,60f1801d8f347996e117f370,google classroom,google classroom,Educational,Educational,PHH


In [124]:
def assign_education_value(row):
    if row['app_name'] in ['aakash','byju','chegg study','diksha','e-pg pathshala',
                           'google classroom', 'microsoft math solver','moodle','mycbseguide', 
                           'physics wallah','simplilearn','vedantu','vidyakul'
                          ]:
#         brainly duolingo
        return 'code_education_app'
    else:
        return 'secondary_education_app'

df_education['app_name_tag'] = df_education.apply(assign_education_value, axis=1)
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
8,60f1801d8f347996e117f370,moodle,moodle,Educational,Educational,PHH,code_education_app
16,60f1801d8f347996e117f370,google classroom,google classroom,Educational,Educational,PHH,code_education_app


In [125]:
df_education[df_education['app_name_tag'] == 'code_education_app'].app_name.unique()

array(['moodle', 'google classroom', 'simplilearn', 'byju',
       'physics wallah', 'aakash', 'diksha', 'chegg study',
       'e-pg pathshala', 'vedantu', 'mycbseguide',
       'microsoft math solver', 'vidyakul'], dtype=object)

In [126]:
edu_ltr_segment_city = df_education\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
edu_ltr_segment_city['% city threshold'] =  (edu_ltr_segment_city['customers']*100.00/edu_ltr_segment_city.customers.sum()).round(2)
edu_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,31852,14.75
1,LTR 0,4425,2.05
2,PHH,179589,83.17
3,UNKNOWN,59,0.03


In [127]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                               \
app_name       aakash bansal classes  brainly     byju caclubindia   
ltr_segment                                                          
HH              152.0            1.0   4467.0   4770.0        14.0   
LTR 0            23.0            NaN    659.0    770.0         2.0   
PHH             604.0            1.0  21736.0  18555.0       144.0   

                                                                        \
app_name    chegg study colegeduniya collegedekho course hero coursera   
ltr_segment                                                              
HH                 94.0         22.0          2.0        10.0   2450.0   
LTR 0              14.0          2.0          NaN         NaN    271.0   
PHH               562.0        117.0          9.0        54.0  18896.0   

                                                                             \
app_name     diksha doubtnut duolingo e-pg pathshala     edx embibe fiitjee   
ltr_segment                                                                   
HH            548.0    518.0   5340.0          100.0   304.0  105.0     NaN   
LTR 0          86.0    124.0    696.0           20.0    41.0   22.0     NaN   
PHH          2133.0   1780.0  36067.0          544.0  2817.0  415.0     1.0   

                                                                \
app_name    geeks for geeks goodreads google classroom happify   
ltr_segment                                                      
HH                    594.0     344.0           8368.0     7.0   
LTR 0                  66.0      53.0           1101.0     1.0   
PHH                  4437.0    4592.0          44542.0    41.0   

                                                                            \
app_name    ignou e-content khan academy meritnation microsoft math solver   
ltr_segment                                                                  
HH                    137.0        183.0         1.0                 112.0   
LTR 0                  15.0         21.0         NaN                  17.0   
PHH                   812.0       1349.0         2.0                 687.0   

                                                                     \
app_name     moodle my study life mycbseguide ncert books photomath   
ltr_segment                                                           
HH            573.0           4.0       116.0       156.0     197.0   
LTR 0          78.0           NaN        20.0        21.0      17.0   
PHH          2889.0          79.0       406.0       860.0    1029.0   

                                                                          \
app_name    physics wallah  pocket shiksha mitra shiksha.com simplilearn   
ltr_segment                                                                
HH                  1401.0   146.0           2.0        44.0       847.0   
LTR 0                196.0    18.0           3.0         8.0        96.0   
PHH                 5627.0  1986.0          14.0       213.0      5603.0   

                                                                              \
app_name    stack overflow  swayam toppr    udemy unacademy vedantu vidyakul   
ltr_segment                                                                    
HH                     7.0  1737.0   4.0   6925.0    2708.0   206.0     17.0   
LTR 0                  3.0   209.0   2.0    817.0     421.0    23.0      2.0   
PHH                   63.0  7325.0  29.0  54780.0   12068.0   858.0     18.0   

                         
app_name    whitehat jr  
ltr_segment              
HH                 72.0  
LTR 0              11.0  
PHH               399.0

In [128]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [129]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                        \
app_name         aakash bansal classes brainly   byju caclubindia chegg study   
ltr_segment                                                                     
HH                 0.48            0.0   14.02  14.98        0.04        0.30   
LTR 0              0.52            NaN   14.89  17.40        0.05        0.32   
PHH                0.34            0.0   12.10  10.33        0.08        0.31   

                                                                            \
app_name    colegeduniya collegedekho course hero coursera diksha doubtnut   
ltr_segment                                                                  
HH                  0.07         0.01        0.03     7.69   1.72     1.63   
LTR 0               0.05          NaN         NaN     6.12   1.94     2.80   
PHH                 0.07         0.01        0.03    10.52   1.19     0.99   

                                                                          \
app_name    duolingo e-pg pathshala   edx embibe fiitjee geeks for geeks   
ltr_segment                                                                
HH             16.77           0.31  0.95   0.33     NaN            1.86   
LTR 0          15.73           0.45  0.93   0.50     NaN            1.49   
PHH            20.08           0.30  1.57   0.23     0.0            2.47   

                                                                             \
app_name    goodreads google classroom happify ignou e-content khan academy   
ltr_segment                                                                   
HH               1.08            26.27    0.02            0.43         0.57   
LTR 0            1.20            24.88    0.02            0.34         0.47   
PHH              2.56            24.80    0.02            0.45         0.75   

                                                                    \
app_name    meritnation microsoft math solver moodle my study life   
ltr_segment                                                          
HH                  0.0                  0.35   1.80          0.01   
LTR 0               NaN                  0.38   1.76           NaN   
PHH                 0.0                  0.38   1.61          0.04   

                                                                     \
app_name    mycbseguide ncert books photomath physics wallah pocket   
ltr_segment                                                           
HH                 0.36        0.49      0.62           4.40   0.46   
LTR 0              0.45        0.47      0.38           4.43   0.41   
PHH                0.23        0.48      0.57           3.13   1.11   

                                                                               \
app_name    shiksha mitra shiksha.com simplilearn stack overflow swayam toppr   
ltr_segment                                                                     
HH                   0.01        0.14        2.66           0.02   5.45  0.01   
LTR 0                0.07        0.18        2.17           0.07   4.72  0.05   
PHH                  0.01        0.12        3.12           0.04   4.08  0.02   

                                                           
app_name     udemy unacademy vedantu vidyakul whitehat jr  
ltr_segment                                                
HH           21.74      8.50    0.65     0.05        0.23  
LTR 0        18.46      9.51    0.52     0.05        0.25  
PHH          30.50      6.72    0.48     0.01        0.22

In [130]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        16069                   20835
LTR 0                      2289                    2782
PHH                       77152                  129876

In [131]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

% Distribution of customers across individual segment.


customers %                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        50.45                   65.41
LTR 0                     51.73                   62.87
PHH                       42.96                   72.32

### Other

#### Service Affinity

In [132]:
service_affinity_city = df_customer_data\
                        .groupby(['service_affinity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
service_affinity_city['% city threshold']=(service_affinity_city['customers']*100.00/service_affinity_city.customers.sum()).round(2)
service_affinity_city.sort_values(by=['service_affinity'],ascending=[True])

,service_affinity,customers,% city threshold
0,AUTO_CAB,28689,2.80
1,LINK_AUTO,101358,9.90
2,LINK_CAB,7221,0.71
3,NO_AFFINITY,97826,9.55
4,ONLY_AUTO,392361,38.31
5,ONLY_CAB,38135,3.72
6,ONLY_LINK,302770,29.56
7,UNKNOWN,55864,5.45


In [133]:
df1 = df_customer_data_explode[~df_customer_data_explode['service_affinity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'service_affinity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,service_affinity_city[['service_affinity','customers']], how= 'left', on='service_affinity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers'])

customers                                                \
categories_l2    Driver_App Educational Finance_Investment Gaming  Office   
service_affinity                                                            
AUTO_CAB               1836        6541              12087   4052   19463   
LINK_AUTO             11402       20701              37346  16002   56744   
LINK_CAB               1192        1310               2802   1119    3883   
NO_AFFINITY           11215       19898              40632  15359   58504   
ONLY_AUTO             25925       89587             136733  57802  243444   
ONLY_CAB               3454        7707              15448   5249   23336   
ONLY_LINK             38709       59374             113888  44206  171339   

                                      
categories_l2       Rest Ride haling  
service_affinity                      
AUTO_CAB           28667       23359  
LINK_AUTO         101329       74210  
LINK_CAB            7219        5069  
NO_AFFINITY        97798       76390  
ONLY_AUTO         392159      304968  
ONLY_CAB           38112       28848  
ONLY_LINK         302661      213984

In [134]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                               \
categories_l2     Driver_App Educational Finance_Investment Gaming Office   
service_affinity                                                            
AUTO_CAB                6.40       22.80              42.13  14.12  67.84   
LINK_AUTO              11.25       20.42              36.85  15.79  55.98   
LINK_CAB               16.51       18.14              38.80  15.50  53.77   
NO_AFFINITY            11.46       20.34              41.53  15.70  59.80   
ONLY_AUTO               6.61       22.83              34.85  14.73  62.05   
ONLY_CAB                9.06       20.21              40.51  13.76  61.19   
ONLY_LINK              12.78       19.61              37.62  14.60  56.59   

                                     
categories_l2      Rest Ride haling  
service_affinity                     
AUTO_CAB          99.92       81.42  
LINK_AUTO         99.97       73.22  
LINK_CAB          99.97       70.20  
NO_AFFINITY       99.97       78.09  
ONLY_AUTO         99.95       77.73  
ONLY_CAB          99.94       75.65  
ONLY_LINK         99.96       70.68

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be LINK Customers.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be AUTO Customers.
    - Educational, Office, Ride haling

#### Distance Preference

In [135]:
distance_affinity_city = df_customer_data\
                        .groupby(['distance_preference'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
distance_affinity_city['% city threshold']=(distance_affinity_city['customers']*100.00/distance_affinity_city.customers.sum()).round(2)
distance_affinity_city.sort_values(by=['distance_preference'],ascending=[True])

,distance_preference,customers,% city threshold
0,LONG_DISTANCE,265711,25.94
1,MEDIUM_DISTANCE,242280,23.65
2,SHORT_DISTANCE,241412,23.57
3,UNKNOWN,274821,26.83


In [136]:
df1 = df_customer_data_explode[~df_customer_data_explode['distance_preference'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'distance_preference'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,distance_affinity_city[['distance_preference','customers']], how= 'left', on='distance_preference')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='distance_preference' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2        Driver_App Educational Finance_Investment Gaming Office   
distance_preference                                                            
LONG_DISTANCE             11.31       17.36              34.28  15.21  53.02   
MEDIUM_DISTANCE            9.21       21.45              37.05  14.39  60.29   
SHORT_DISTANCE             7.34       24.97              39.19  14.46  65.19   

                                        customers              \
categories_l2         Rest Ride haling Driver_App Educational   
distance_preference                                             
LONG_DISTANCE        99.96       71.02    30053.0     46116.0   
MEDIUM_DISTANCE      99.95       75.36    22320.0     51973.0   
SHORT_DISTANCE       99.95       78.53    17720.0     60272.0   

                                                                     \
categories_l2       Finance_Investment   Gaming    Office      Rest   
distance_preference                                                   
LONG_DISTANCE                  91091.0  40418.0  140879.0  265601.0   
MEDIUM_DISTANCE                89755.0  34870.0  146061.0  242167.0   
SHORT_DISTANCE                 94617.0  34898.0  157378.0  241302.0   

                                 
categories_l2       Ride haling  
distance_preference              
LONG_DISTANCE          188711.0  
MEDIUM_DISTANCE        182585.0  
SHORT_DISTANCE         189588.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to do LONG Distance rides.
    - Driver_App, Gaming
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do MEDIUM Distance rides.
    - Finance_Investment
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do SHORT Distance rides.
    - Educational, Office

#### Gender

In [137]:
gender_city = df_customer_data\
                        .groupby(['gender'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
gender_city['% city threshold']=(gender_city['customers']*100.00/gender_city.customers.sum()).round(2)
gender_city.sort_values(by=['gender'],ascending=[True])

,gender,customers,% city threshold
0,FEMALE,297281,29.02
1,MALE,691917,67.56
2,OTHERS,3494,0.34
3,UNKNOWN,31532,3.08


In [138]:
df1 = df_customer_data_explode[~df_customer_data_explode['gender'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'gender'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,gender_city[['gender','customers']], how= 'left', on='gender')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='gender' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
gender                                                                          
FEMALE               3.72       24.43              31.82  16.32  66.66  99.95   
MALE                12.65       19.59              39.21  14.27  56.11  99.96   
OTHERS               8.96       15.11              13.48  17.03  33.03  99.91   

                           customers                                          \
categories_l2 Ride haling Driver_App Educational Finance_Investment   Gaming   
gender                                                                         
FEMALE              82.07    11055.0     72616.0            94587.0  48523.0   
MALE                71.35    87524.0    135552.0           271278.0  98750.0   
OTHERS              57.90      313.0       528.0              471.0    595.0   

                                               
categories_l2    Office      Rest Ride haling  
gender                                         
FEMALE         198177.0  297137.0    243978.0  
MALE           388241.0  691640.0    493678.0  
OTHERS           1154.0    3491.0      2023.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be MALE.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be FEMALE.
    - Educational

#### Income Segment

In [139]:
income_segment_city = df_customer_data\
                        .groupby(['income_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
income_segment_city['% city threshold']=(income_segment_city['customers']*100.00/income_segment_city.customers.sum()).round(2)
income_segment_city.sort_values(by=['income_segment'],ascending=[True])

,income_segment,customers,% city threshold
0,HIGH_INCOME,441713,43.13
1,LOW_INCOME,61040,5.96
2,MEDIUM_INCOME,359479,35.10
3,UNKNOWN,161992,15.82


In [140]:
df1 = df_customer_data_explode[~df_customer_data_explode['income_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'income_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,income_segment_city[['income_segment','customers']], how= 'left', on='income_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='income_segment' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2   Driver_App Educational Finance_Investment Gaming Office   
income_segment                                                            
HIGH_INCOME           9.79       25.46              46.22  16.60  68.94   
LOW_INCOME            7.75        9.76              16.22  10.15  32.91   
MEDIUM_INCOME         9.83       20.13              30.65  14.40  56.24   

                                   customers                                 \
categories_l2    Rest Ride haling Driver_App Educational Finance_Investment   
income_segment                                                                
HIGH_INCOME     99.98       83.37    43227.0    112472.0           204158.0   
LOW_INCOME      99.85       51.25     4729.0      5957.0             9898.0   
MEDIUM_INCOME   99.95       71.22    35324.0     72368.0           110182.0   

                                                         
categories_l2    Gaming    Office      Rest Ride haling  
income_segment                                           
HIGH_INCOME     73306.0  304504.0  441607.0    368241.0  
LOW_INCOME       6198.0   20091.0   60948.0     31285.0  
MEDIUM_INCOME   51778.0  202174.0  359299.0    256032.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be HIGH_INCOME.
    - Office, Educational, Finance_Investment, Gaming, Ride haling
- Whenever an app belongs to the app-categories listed below, customers are more likely to be MEDIUM_INCOME.
    - Driver_App

#### Price Sensitivity

In [141]:
ps_city = df_customer_data\
                        .groupby(['price_sensitivity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ps_city['% city threshold']=(ps_city['customers']*100.00/ps_city.customers.sum()).round(2)
ps_city.sort_values(by=['price_sensitivity'],ascending=[True])

,price_sensitivity,customers,% city threshold
0,NPS,353345,34.5
1,PS,283735,27.7
2,UNKNOWN,387144,37.8


In [142]:
df1 = df_customer_data_explode[~df_customer_data_explode['price_sensitivity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'price_sensitivity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ps_city[['price_sensitivity','customers']], how= 'left', on='price_sensitivity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='price_sensitivity' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2      Driver_App Educational Finance_Investment Gaming Office   
price_sensitivity                                                            
NPS                      8.63       22.45              38.59  14.97  62.89   
PS                       9.60       21.15              37.71  15.27  59.25   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
price_sensitivity                                             
NPS                99.96       78.91    30499.0     79323.0   
PS                 99.96       77.39    27248.0     60019.0   

                                                                               
categories_l2     Finance_Investment   Gaming    Office      Rest Ride haling  
price_sensitivity                                                              
NPS                         136341.0  52910.0  222223.0  353199.0    278834.0  
PS                          107005.0  43336.0  168121.0  283625.0    219582.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be Price Sensitivity.
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers are less likely to be Price Sensitivity. (Less confidence - Uninterpretable/Hard to interpretable using Price Sensitivity)
    - Office, Educational, Finance_Investment, Ride haling, Gaming

#### RPC

In [143]:
rpc_bucket_city = df_customer_data\
                        .groupby(['rpc_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rpc_bucket_city['% city threshold']=(rpc_bucket_city['customers']*100.00/rpc_bucket_city.customers.sum()).round(2)
rpc_bucket_city.sort_values(by=['rpc_bucket'],ascending=[True])

,rpc_bucket,customers,% city threshold
0,a. ZERO,176004,17.18
1,b. LOW,377023,36.81
2,c. MEDIUM,273959,26.75
3,d. HIGH,197215,19.26


In [144]:
df1 = df_customer_data_explode[~df_customer_data_explode['rpc_bucket'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'rpc_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,rpc_bucket_city[['rpc_bucket','customers']], how= 'left', on='rpc_bucket')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='rpc_bucket' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
rpc_bucket                                                                      
a. ZERO             11.68       22.21              37.03  15.86  58.63  99.96   
b. LOW              10.76       20.28              35.78  15.07  57.21  99.96   
c. MEDIUM            9.22       20.94              37.15  14.66  59.75  99.96   
d. HIGH              7.25       21.80              38.16  14.13  62.83  99.95   

                           customers                                          \
categories_l2 Ride haling Driver_App Educational Finance_Investment   Gaming   
rpc_bucket                                                                     
a. ZERO             75.51    20555.0     39096.0            65176.0  27909.0   
b. LOW              71.41    40574.0     76469.0           134902.0  56817.0   
c. MEDIUM           74.42    25255.0     57357.0           101779.0  40164.0   
d. HIGH             79.94    14299.0     42998.0            75267.0  27874.0   

                                               
categories_l2    Office      Rest Ride haling  
rpc_bucket                                     
a. ZERO        103197.0  175929.0    132896.0  
b. LOW         215705.0  376858.0    269217.0  
c. MEDIUM      163703.0  273843.0    203889.0  
d. HIGH        123918.0  197126.0    157654.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers having Less RPC
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers having Good RPC
    - Educational, Finance_Investment, Office, Ride haling

#### App Count Bucket

In [145]:
app_count_bucket_city = df_customer_data\
                        .groupby(['app_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
app_count_bucket_city['% city threshold']=(app_count_bucket_city['customers']*100.00/app_count_bucket_city.customers.sum()).round(2)
app_count_bucket_city.sort_values(by=['app_count_bucket'],ascending=[True])

,app_count_bucket,customers,% city threshold
0,1-5,882765,86.19
1,11-15,28357,2.77
2,6-10,99968,9.76
3,Above 15,13134,1.28


In [146]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'app_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'app_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='app_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2    Driver_App Educational Finance_Investment Gaming Office   
app_count_bucket                                                           
1-5                   90.30       85.63              85.65  87.03  85.24   
11-15                  1.86        2.94               2.90   2.56   3.01   
6-10                   7.02       10.20              10.22   9.25  10.44   
Above 15               0.82        1.23               1.22   1.16   1.31   

                                     
categories_l2      Rest Ride haling  
app_count_bucket                     
1-5               86.19       85.01  
11-15              2.77        3.07  
6-10               9.76       10.49  
Above 15           1.28        1.43

In [147]:
df_customer_data.app_count.describe()

count    1.024224e+06
mean     1.973654e+01
std      9.985884e+00
min      1.000000e+00
25%      1.200000e+01
50%      1.800000e+01
75%      2.600000e+01
max      8.400000e+01
Name: app_count, dtype: float64

#### Category Count Bucket

In [148]:
cat_count_bucket_city = df_customer_data\
                        .groupby(['category_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'),)\
                        .reset_index()
cat_count_bucket_city['% city threshold']=(cat_count_bucket_city['customers']*100.00/cat_count_bucket_city.customers.sum()).round(2)
cat_count_bucket_city.sort_values(by=['category_count_bucket'],ascending=[True])

,category_count_bucket,customers,% city threshold
0,1,108207,10.56
1,2,214069,20.90
2,3,279069,27.25
3,Above 3,422879,41.29


In [149]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'category_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'category_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='category_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2         Driver_App Educational Finance_Investment Gaming Office   
category_count_bucket                                                           
1                           0.02        0.00               0.01   0.01   0.01   
2                          13.30        2.97               4.00   9.37   7.12   
3                          28.92       12.24              15.94  22.09  29.05   
Above 3                    57.77       84.79              80.05  68.53  63.82   

                                          
categories_l2           Rest Ride haling  
category_count_bucket                     
1                      10.54        0.02  
2                      20.90       15.95  
3                      27.26       30.46  
Above 3                41.31       53.58

In [150]:
df_temp = df_customer_data[df_customer_data['category_count_bucket'].isin(['2-3'])]\
.groupby(['customer_id'])\
.agg({'categories_l2' : set})\
.reset_index()
df_temp[['categories_l2']]

,categories_l2


In [151]:
df_customer_data.category_count.describe()

count    1.024224e+06
mean     3.163860e+00
std      1.258072e+00
min      1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      7.000000e+00
Name: category_count, dtype: float64

#### Customer Use-Case 

In [152]:
usecase_tag_city = df_customer_data\
                        .groupby(['usecase_tag'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
usecase_tag_city['% city threshold']=(usecase_tag_city['customers']*100.00/usecase_tag_city.customers.sum()).round(2)
usecase_tag_city.sort_values(by=['usecase_tag'],ascending=[True])

,usecase_tag,customers,% city threshold
0,Unknown,71858,7.02
1,educational,63446,6.19
2,food,42310,4.13
3,govt_amenity,12482,1.22
4,health_and_personal,59262,5.79
5,household_needs,124483,12.15
6,leisure,100580,9.82
7,office,116942,11.42
8,place_of_worship,15914,1.55
9,residential,264611,25.84


In [153]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'usecase_tag'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'usecase_tag'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='usecase_tag' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2       Driver_App Educational Finance_Investment Gaming Office   
usecase_tag                                                                   
Unknown                   8.41        6.45               6.70   7.16   6.55   
educational               6.33        6.49               5.23   6.23   5.90   
food                      3.86        4.32               4.21   4.27   4.23   
govt_amenity              1.25        1.17               1.22   1.24   1.21   
health_and_personal       6.31        5.01               5.02   5.78   5.32   
household_needs          11.81       12.22              12.33  12.21  12.38   
leisure                   9.51        9.81               9.44   9.75   9.77   
office                    9.66       12.92              13.86  11.35  12.96   
place_of_worship          1.65        1.40               1.36   1.58   1.45   
residential              24.92       26.07              26.15  25.33  26.17   
transit_station          16.28       14.13              14.48  15.12  14.06   

                                        
categories_l2         Rest Ride haling  
usecase_tag                             
Unknown               7.02        6.62  
educational           6.19        6.19  
food                  4.13        4.19  
govt_amenity          1.22        1.22  
health_and_personal   5.79        5.68  
household_needs      12.15       12.25  
leisure               9.82        9.98  
office               11.42       12.02  
place_of_worship      1.55        1.52  
residential          25.83       26.44  
transit_station      14.87       13.88

#### AO-NET Funnel

In [154]:
df_customer_data['city'] = 'Bangalore, Android'
city_funnel = df_customer_data\
                        .groupby(['city'])\
                        .agg(fe_count = ('fe_count','sum'),
                             rr_count = ('rr_count','sum'),
                             net_count = ('net_count','sum')
                            )\
                        .reset_index()

city_funnel['% fe2rr']=(city_funnel['rr_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel['% g2n']=(city_funnel['net_count']*100.00/city_funnel['rr_count']).round(2)
city_funnel['% fe2net']=(city_funnel['net_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel[['city','% fe2rr','% g2n','% fe2net']].T

,0
city,"Bangalore, Android"
% fe2rr,45.91
% g2n,52.8
% fe2net,24.24


In [155]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y'])\
.agg(customers = ('customer_id','nunique'),
     fe_count = ('fe_count','sum'),
     rr_count = ('rr_count','sum'),
     net_count = ('net_count','sum')
    )\
.sort_values(by=['categories_l2_y'],ascending=[False])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['% fe2rr']=(df2['rr_count']*100.00/df2['fe_count']).round(2)
df2['% g2n']=(df2['net_count']*100.00/df2['rr_count']).round(2)
df2['% fe2net']=(df2['net_count']*100.00/df2['fe_count']).round(2)
df2[['categories_l2','% fe2rr','% g2n','% fe2net']].T

,0,1,2,3,4,5,6
categories_l2,Rest,Ride haling,Office,Finance_Investment,Educational,Gaming,Driver_App
% fe2rr,45.91,46.78,46.56,45.8,46.04,45.64,45.04
% g2n,52.8,52.59,52.85,51.85,51.87,51.39,50.07
% fe2net,24.24,24.6,24.61,23.75,23.88,23.46,22.55
